# Optimize a RAG Application with AutoRAG Pipelines

This notebook runs from an **in-cluster Workbench**. It uses Python clients for object storage, Kubernetes, and Kubeflow Pipelines, so all operations run directly from the Workbench without a local port-forward.

AutoRAG searches retrieval-augmented generation (RAG) configurations for a document corpus. The workflow can discover documents, extract text with Docling, prepare a search space, optimize RAG templates with ai4rag, and index the selected production corpus.

This notebook demonstrates:

1. Uploading a small corpus and benchmark to S3-compatible storage with `boto3`.
2. Creating the input and OGX Kubernetes Secrets with the Kubernetes Python client.
3. Compiling and submitting the ingestion pipeline from the Workbench.
4. Optionally submitting the full optimization and indexing pipelines.

> The current `odh-autorag:odh-stable` image ships ai4rag 0.17 and is used by the ingestion example below. The managed optimization and indexing wrappers in `pipelines-components` currently import ai4rag 0.10 APIs. The full optimization path therefore requires a runtime image rebuilt from the repository `Containerfile` with ai4rag `~=0.10.1`, plus OGX and a vector store.


## Prerequisites

- A Workbench running **inside the cluster** and able to reach the KFP, S3, and OGX Services.
- Kubeflow Pipelines 2.16.1 through either the shared `kfp-operator` service or DSPO.
- An S3-compatible bucket. The input bucket can be different from the KFP artifact store.
- Permission for the Workbench identity to create or update Secrets in the pipeline namespace, or pre-created Secrets named by `INPUT_SECRET_NAME` and `OGX_SECRET_NAME`.
- For optimization and indexing: an OGX endpoint with embedding and generation models and a registered vector I/O provider.

The default values below match the x86 development environment. Change them for another cluster.


In [ ]:
# Install the SDKs into the Workbench kernel.
# Run this cell once after starting a new Workbench.
%pip install --quiet 'kfp==2.16.1' 'kfp-kubernetes==2.16.1' boto3 kubernetes


In [ ]:
import json
import os
import sys
import subprocess
from pathlib import Path

# Required values can be set in the Workbench environment before running the notebook.
# Do not commit real credentials into the notebook.
NAMESPACE = os.environ.get("AUTORAG_NAMESPACE", "my-pipeline-namespace")
S3_ENDPOINT = os.environ.get(
    "AUTORAG_S3_ENDPOINT", "http://seaweedfs.mlops-demo-e2e:8333"
)
S3_BUCKET = os.environ.get("AUTORAG_S3_BUCKET", "mlpipeline")
S3_ACCESS_KEY = os.environ.get("AWS_ACCESS_KEY_ID", "minio")
S3_SECRET_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "minio123")
S3_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")

# Use the in-cluster KFP Service. For a DSPO instance named `sample`, use:
# http://ds-pipeline-sample.<namespace>.svc:8888
KFP_HOST = os.environ.get(
    "KFP_HOST", "http://ml-pipeline.kubeflow.svc:8888"
)

# OGX values are only needed for the full optimization/indexing examples.
OGX_BASE_URL = os.environ.get(
    "OGX_CLIENT_BASE_URL", "http://llama-stack.my-project.svc:8321"
)
OGX_API_KEY = os.environ.get("OGX_CLIENT_API_KEY", "not-used")
VECTOR_IO_PROVIDER_ID = os.environ.get("AUTORAG_VECTOR_IO_PROVIDER_ID", "milvus-remote")

INPUT_SECRET_NAME = os.environ.get("AUTORAG_INPUT_SECRET_NAME", "autorag-input-s3")
OGX_SECRET_NAME = os.environ.get("AUTORAG_OGX_SECRET_NAME", "autorag-ogx")
PREFIX = os.environ.get("AUTORAG_PREFIX", "autorag-workbench-demo")
DOCUMENT_KEY = f"{PREFIX}/documents/acp-rag-overview.md"
BENCHMARK_KEY = f"{PREFIX}/benchmark.json"

print({
    "namespace": NAMESPACE,
    "s3_endpoint": S3_ENDPOINT,
    "s3_bucket": S3_BUCKET,
    "kfp_host": KFP_HOST,
    "document_key": DOCUMENT_KEY,
    "benchmark_key": BENCHMARK_KEY,
})


## Upload the corpus with `boto3`

The benchmark uses `correct_answer_document_keys`, which is the field expected by ai4rag 0.17. The keys are relative document names; AutoRAG matches them against the S3 object keys and file names during discovery.


In [ ]:
import boto3
from botocore.config import Config

DOCUMENT = '''# AutoRAG on Alauda AI

AutoRAG searches chunking, embedding, retrieval, and generation settings for a document corpus.
It evaluates candidate RAG patterns with a benchmark and writes a deployable pattern.json.

The production workflow uses the selected pattern's indexing settings to build a vector collection.
'''

BENCHMARK = [
    {
        "question": "What does AutoRAG search?",
        "correct_answers": ["Chunking, embedding, retrieval, and generation settings"],
        "correct_answer_document_keys": ["acp-rag-overview.md"],
    },
    {
        "question": "Which file contains deployable RAG settings?",
        "correct_answers": ["pattern.json"],
        "correct_answer_document_keys": ["acp-rag-overview.md"],
    },
]

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=S3_ACCESS_KEY,
    aws_secret_access_key=S3_SECRET_KEY,
    region_name=S3_REGION,
    config=Config(s3={"addressing_style": "path"}),
)

# The bucket is normally provisioned by the platform. Uncomment this only for
# a standalone MinIO/SeaweedFS installation where the bucket does not exist.
# s3.create_bucket(Bucket=S3_BUCKET)

s3.put_object(
    Bucket=S3_BUCKET,
    Key=DOCUMENT_KEY,
    Body=DOCUMENT.encode("utf-8"),
    ContentType="text/markdown",
)
s3.put_object(
    Bucket=S3_BUCKET,
    Key=BENCHMARK_KEY,
    Body=json.dumps(BENCHMARK, indent=2).encode("utf-8"),
    ContentType="application/json",
)

objects = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=PREFIX).get("Contents", [])
print([(obj["Key"], obj["Size"]) for obj in objects])


## Create the Kubernetes Secrets with the Python client

Secrets are namespace-scoped. The pipeline run and these Secrets must use the same namespace. The cell uses in-cluster authentication from the Workbench; it falls back to the local kubeconfig when the notebook is run outside Kubernetes.


In [ ]:
from kubernetes import client, config
from kubernetes.client.rest import ApiException

try:
    config.load_incluster_config()
except config.ConfigException:
    config.load_kube_config()

core = client.CoreV1Api()

def upsert_secret(name, data):
    secret = client.V1Secret(
        metadata=client.V1ObjectMeta(name=name),
        type="Opaque",
        string_data=data,
    )
    try:
        core.read_namespaced_secret(name=name, namespace=NAMESPACE)
        core.replace_namespaced_secret(name=name, namespace=NAMESPACE, body=secret)
        print(f"updated Secret {NAMESPACE}/{name}")
    except ApiException as exc:
        if exc.status != 404:
            raise
        core.create_namespaced_secret(namespace=NAMESPACE, body=secret)
        print(f"created Secret {NAMESPACE}/{name}")

upsert_secret(INPUT_SECRET_NAME, {
    "AWS_ACCESS_KEY_ID": S3_ACCESS_KEY,
    "AWS_SECRET_ACCESS_KEY": S3_SECRET_KEY,
    "AWS_S3_ENDPOINT": S3_ENDPOINT,
    "AWS_DEFAULT_REGION": S3_REGION,
})

upsert_secret(OGX_SECRET_NAME, {
    "OGX_CLIENT_BASE_URL": OGX_BASE_URL,
    "OGX_CLIENT_API_KEY": OGX_API_KEY,
})


## Install the pipeline components and compiler helper

Set `PIPELINES_COMPONENTS_DIR` to a checkout of the Alauda `pipelines-components` mirror. Set `AML_DOCS_DIR` to a checkout of this documentation repository if the compiler helper is not already available in the Workbench.

The helper compiles an ingestion DAG that uses ai4rag 0.17 APIs and also contains optional optimization/indexing variants for a runtime rebuilt with ai4rag 0.10.1.


In [ ]:
PIPELINES_COMPONENTS_DIR = Path(
    os.environ.get("PIPELINES_COMPONENTS_DIR", "/home/jovyan/pipelines-components")
)
AML_DOCS_DIR = Path(os.environ.get("AML_DOCS_DIR", "/home/jovyan/aml-docs"))
COMPILER = AML_DOCS_DIR / "docs/en/train/guides/assets/autorag/compile_autorag_pipelines.py"

if not PIPELINES_COMPONENTS_DIR.exists():
    raise FileNotFoundError(
        f"Clone the pipelines-components mirror into {PIPELINES_COMPONENTS_DIR} "
        "or set PIPELINES_COMPONENTS_DIR."
    )
if not COMPILER.exists():
    raise FileNotFoundError(
        f"Clone aml-docs into {AML_DOCS_DIR} or set AML_DOCS_DIR so the notebook "
        "can find {COMPILER}."
    )

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e",
    str(PIPELINES_COMPONENTS_DIR),
])
print("compiler:", COMPILER)


In [ ]:
COMPILED_DIR = Path.cwd() / "autorag-compiled"
COMPILED_DIR.mkdir(exist_ok=True)
AUTORAG_IMAGE = os.environ.get(
    "RELATED_IMAGE_ODH_AUTORAG_IMAGE",
    "quay.io/opendatahub/odh-autorag:odh-stable",
)

# Compile only the ai4rag 0.17-compatible ingestion example by default.
env = os.environ.copy()
env["RELATED_IMAGE_ODH_AUTORAG_IMAGE"] = AUTORAG_IMAGE
subprocess.check_call([
    sys.executable, str(COMPILER), "ingestion",
    "--output-dir", str(COMPILED_DIR),
], env=env)
print(COMPILED_DIR / "ingestion.yaml")


## Submit the ingestion pipeline from the Workbench

The KFP Service is addressed directly inside the cluster. For DSPO, replace `KFP_HOST` with the DSPA Service URL shown in the comment above.


In [ ]:
from kfp.client import Client

client = Client(host=KFP_HOST, namespace=NAMESPACE)
run = client.create_run_from_pipeline_package(
    str(COMPILED_DIR / "ingestion.yaml"),
    arguments={
        "test_data_secret_name": INPUT_SECRET_NAME,
        "test_data_bucket_name": S3_BUCKET,
        "test_data_key": BENCHMARK_KEY,
        "input_data_secret_name": INPUT_SECRET_NAME,
        "input_data_bucket_name": S3_BUCKET,
        "input_data_key": f"{PREFIX}/documents",
        "preset": "speed",
    },
    enable_caching=False,
)
print("run_id:", run.run_id)


In [ ]:
# Wait for the ingestion smoke run and print its final state.
finished = client.wait_for_run_completion(run.run_id, timeout=3600)
print(finished)


A successful ingestion run produces these artifacts:

| Task | Artifact | Meaning |
|---|---|---|
| `test-data-loader` | `test_data` | Validated benchmark JSON |
| `documents-discovery` | `discovered_documents` | Document descriptor with bucket, prefix, and file list |
| `text-extraction` | `extracted_text` | DoclingDocument JSON files |

Resolve ingestion problems before spending OGX inference on optimization.


## Submit full RAG optimization (optional)

Set `RUN_FULL_OPTIMIZATION = True` only after confirming that:

- the runtime image was rebuilt with ai4rag `~=0.10.1` for the current `pipelines-components` wrappers;
- the OGX Secret is reachable from the pipeline namespace;
- OGX advertises an embedding model and generation model; and
- `VECTOR_IO_PROVIDER_ID` is registered in OGX.

The default `odh-autorag:odh-stable` image is not sufficient for this path because it currently ships ai4rag 0.17.


In [ ]:
RUN_FULL_OPTIMIZATION = False

if RUN_FULL_OPTIMIZATION:
    env = os.environ.copy()
    env["RELATED_IMAGE_ODH_AUTORAG_IMAGE"] = AUTORAG_IMAGE
    subprocess.check_call([
        sys.executable, str(COMPILER), "optimization", "indexing",
        "--output-dir", str(COMPILED_DIR),
    ], env=env)

    optimization_run = client.create_run_from_pipeline_package(
        str(COMPILED_DIR / "optimization.yaml"),
        arguments={
            "test_data_secret_name": INPUT_SECRET_NAME,
            "test_data_bucket_name": S3_BUCKET,
            "test_data_key": BENCHMARK_KEY,
            "input_data_secret_name": INPUT_SECRET_NAME,
            "input_data_bucket_name": S3_BUCKET,
            "input_data_key": f"{PREFIX}/documents",
            "ogx_secret_name": OGX_SECRET_NAME,
            "vector_io_provider_id": VECTOR_IO_PROVIDER_ID,
            "optimization_metric": "overall_score",
            "optimization_max_rag_patterns": 2,
            "preset": "speed",
        },
        enable_caching=False,
    )
    print("optimization run_id:", optimization_run.run_id)
else:
    print("Skipped. Set RUN_FULL_OPTIMIZATION=True after validating OGX and the ai4rag 0.10 runtime.")


## Index the production corpus (optional)

After selecting a ranked `pattern.json`, copy its `embedding_model_id`, `vector_store_id`, `chunking_method`, `chunk_size`, and `chunk_overlap` into the indexing pipeline. The same OGX Secret and vector I/O provider must be available.


In [ ]:
if RUN_FULL_OPTIMIZATION:
    indexing_run = client.create_run_from_pipeline_package(
        str(COMPILED_DIR / "indexing.yaml"),
        arguments={
            "ogx_secret_name": OGX_SECRET_NAME,
            "embedding_model_id": os.environ["AUTORAG_EMBEDDING_MODEL_ID"],
            "vector_io_provider_id": VECTOR_IO_PROVIDER_ID,
            "input_data_secret_name": INPUT_SECRET_NAME,
            "input_data_bucket_name": S3_BUCKET,
            "input_data_key": f"{PREFIX}/documents",
            "chunking_method": "recursive",
            "chunk_size": 512,
            "chunk_overlap": 64,
            "distance_metric": "cosine",
            "batch_size": 20,
        },
        enable_caching=False,
    )
    print("indexing run_id:", indexing_run.run_id)
else:
    print("Skipped because full optimization is disabled.")


## Troubleshooting

- **Secret or S3 errors:** confirm the Workbench can resolve `S3_ENDPOINT`, the bucket exists, and the Secret is in `NAMESPACE`.
- **`No module named 'ai4rag.components'`:** use the ai4rag 0.17 ingestion components in this notebook, or rebuild the runtime from the `pipelines-components` Containerfile with ai4rag 0.10.1 for the managed optimization wrappers.
- **`No matching distribution found for kfp==2.16.1`:** compile with `install_kfp_package=False`; the runtime already contains the SDK.
- **Optimization remains pending:** check CPU and memory requests, OGX reachability, model availability, and vector-provider registration.
